In [10]:
!py -m pip install pandas openpyxl xlrd -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
 
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

### Convert xls to csv data

In [ ]:
df = pd.read_excel("data/macro/DataWeb-Query-Export.xlsx", sheet_name="Query Results")
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "Year": "year",
    "Quarter": "quarter",
    df.columns[-1]: "trade_balance"
})

df = df.dropna(subset=["year", "quarter"])
df["year"] = df["year"].astype(int)
df["quarter"] = df["quarter"].astype(int)

df["date"] = pd.PeriodIndex.from_fields(year=df["year"], quarter=df["quarter"], freq="Q").to_timestamp("Q")
df = df[["date", "trade_balance"]]

output_path = "data/macro/dataweb.csv"
df.to_csv(output_path, index=False)
print(f"Converted: data/macro/DataWeb-Query-Export.xlsx → {output_path}")

In [ ]:
df = pd.read_excel("data/macro/gscpi_data.xls", sheet_name = "GSCPI Monthly Data")
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
df = df.dropna(subset=["Date"])
df.to_csv("data/macro/gscpi.csv", index=False)
print(f"Converted!")


### Merge CSVs

In [63]:
 
def to_quarter_end(series: pd.Series) -> pd.Series:
    return  series.dt.to_period("Q").dt.to_timestamp("Q")
 

In [91]:
crisp = pd.read_csv("data/macro/crisp indices.csv", parse_dates=["QTDATE"])
crisp["date"] = to_quarter_end(crisp["QTDATE"])

crisp = (
    crisp.groupby("date")
    .agg(YLDMAT=("YLDMAT", "last"), DAYMAT=("DAYMAT", "last"))
    .reset_index()
)

treasury = pd.read_csv("data/macro/crisp treasury and inflation.csv", parse_dates=["caldt"])
treasury = treasury.rename(columns={"caldt": "date"})
treasury["date"] = to_quarter_end(treasury["date"])


merged = pd.merge(crisp, treasury, on="date", how="outer")
merged = merged.sort_values("date").reset_index(drop=True)
merged.shape

(44, 23)

In [92]:
wti = pd.read_csv("data/macro/Cushing_OK_WTI_Spot_Price_FOB.csv", skiprows=4, parse_dates=["Month"])
wti = wti.rename(columns={"Cushing OK WTI Spot Price FOB Dollars per Barrel": "wti_price"})
wti["date"] = to_quarter_end(wti["Month"])
wti = wti.sort_values("date")
wti = (
    wti.groupby("date")
    .agg(wti_price=("wti_price", "last"))
    .reset_index()
)

merged = pd.merge(merged, wti, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)

print(merged.shape)
print(merged.tail())


(44, 24)
         date  YLDMAT  DAYMAT    b30ret    b30ind    b20ret    b20ind  \
39 2023-12-31   3.869    1494  0.126552  2439.566  0.107530  3087.850   
40 2024-03-31   4.184    1770 -0.040914  2339.754 -0.026131  3007.162   
41 2024-06-30   4.365    1678 -0.023592  2284.554 -0.018120  2952.674   
42 2024-09-30   3.540    1584  0.084607  2477.843  0.078831  3185.438   
43 2024-12-31   4.313    1492 -0.094580  2243.488 -0.079826  2931.156   

      b10ret    b10ind     b7ret  ...     b2ind     b1ret     b1ind    t90ret  \
39  0.067224  2430.371  0.055674  ...  1584.475  0.019206  1398.038  0.013657   
40 -0.016383  2390.554 -0.014760  ...  1587.876  0.007759  1408.885  0.013044   
41 -0.004645  2379.449 -0.001042  ...  1601.139  0.011000  1424.383  0.013536   
42  0.059990  2522.193  0.057678  ...  1652.608  0.022007  1455.729  0.014198   
43 -0.051607  2392.029 -0.040356  ...  1649.763  0.007522  1466.680  0.011852   

      t90ind    t30ret    t30ind    cpiret  cpiind  wti_price  
3

In [ ]:
# Trade Balance
dataweb = pd.read_csv("data/macro/dataweb.csv", parse_dates=["date"])
dataweb["date"] = to_quarter_end(dataweb["date"])
merged = pd.merge(merged, dataweb, on="date", how="left")

# ECI
eci = pd.read_csv("data/macro/ECIALLCIV.csv", parse_dates=["observation_date"])
eci = eci.rename(columns={"observation_date": "date", "ECIALLCIV": "eci"})
eci["date"] = to_quarter_end(eci["date"])
eci = eci.groupby("date").agg(eci=("eci", "last")).reset_index()
merged = pd.merge(merged, eci, on="date", how="left")

#  GDP 
gdp = pd.read_csv("data/macro/GDP.csv", parse_dates=["observation_date"])
gdp = gdp.rename(columns={"observation_date": "date", "GDP": "gdp"})
gdp["date"] = to_quarter_end(gdp["date"])
gdp = gdp.groupby("date").agg(gdp=("gdp", "last")).reset_index()
merged = pd.merge(merged, gdp, on="date", how="left")

# GSCPI
gscpi = pd.read_csv("data/macro/gscpi.csv", parse_dates=["Date"])
gscpi = gscpi[["Date", "GSCPI"]].rename(columns={"Date": "date", "GSCPI": "gscpi"})
gscpi["date"] = to_quarter_end(gscpi["date"])
gscpi = gscpi.groupby("date").agg(gscpi=("gscpi", "mean")).reset_index()
merged = pd.merge(merged, gscpi, on="date", how="left")

# Interest Rates
ir = pd.read_csv("data/macro/Interest rates.csv", parse_dates=["observation_date"])
ir = ir.rename(columns={"observation_date": "date", "IRLTLT01USQ156N": "long_term_rate"})
ir["date"] = to_quarter_end(ir["date"])
ir = ir.groupby("date").agg(long_term_rate=("long_term_rate", "mean")).reset_index()
merged = pd.merge(merged, ir, on="date", how="left")

merged = merged.sort_values("date").reset_index(drop=True)


In [ ]:
medcpi = pd.read_csv("data/macro/MEDCPI_US.csv")
medcpi["date"] = pd.to_datetime(medcpi["observation_date"], format="%d-%m-%Y")
medcpi = medcpi.rename(columns={"MEDCPIM158SFRBCLE": "median_cpi"})
medcpi["date"] = to_quarter_end(medcpi["date"])
medcpi["date"] = medcpi["date"] + pd.offsets.QuarterEnd(1)

medcpi = medcpi.groupby("date").agg(median_cpi=("median_cpi", "mean")).reset_index()
merged = pd.merge(merged, medcpi, on="date", how="left")

In [96]:
nfci = pd.read_csv("data/macro/nfci-data-series-csv.csv", parse_dates=["Friday_of_Week"])
nfci = nfci.rename(columns={"Friday_of_Week": "date"})
nfci["date"] = to_quarter_end(nfci["date"])
nfci = nfci.groupby("date").agg(
    nfci=("NFCI", "mean"),
    anfci=("ANFCI", "mean"),
    nfci_risk=("Risk", "mean"),
    nfci_credit=("Credit", "mean"),
    nfci_leverage=("Leverage", "mean"),
    nfci_nonfinancial_leverage=("Nonfinancial_Leverage", "mean")
).reset_index()

merged = pd.merge(merged, nfci, on="date", how="left")
merged = merged.sort_values("date").reset_index(drop=True)

(44, 36)
date                          0
YLDMAT                        0
DAYMAT                        0
b30ret                        0
b30ind                        0
b20ret                        0
b20ind                        0
b10ret                        0
b10ind                        0
b7ret                         0
b7ind                         0
b5ret                         0
b5ind                         0
b2ret                         0
b2ind                         0
b1ret                         0
b1ind                         0
t90ret                        0
t90ind                        0
t30ret                        0
t30ind                        0
cpiret                        0
cpiind                        0
wti_price                     0
trade_balance                 0
eci                           0
gdp                           0
gscpi                         0
long_term_rate                0
median_cpi                    0
nfci                          0

In [97]:
# --- PPI (quarterly but starts Jan, shift forward one quarter like medcpi) ---
ppi = pd.read_csv("data/macro/PPIACO.csv", parse_dates=["observation_date"])
ppi = ppi.rename(columns={"observation_date": "date", "PPIACO": "ppi"})
ppi["date"] = to_quarter_end(ppi["date"])
ppi["date"] = ppi["date"] + pd.offsets.QuarterEnd(1)
ppi = ppi.groupby("date").agg(ppi=("ppi", "last")).reset_index()
merged = pd.merge(merged, ppi, on="date", how="left")

# --- Real Residential Property Prices (same offset issue) ---
prop = pd.read_csv("data/macro/real residential property prices.csv", parse_dates=["observation_date"])
prop = prop.rename(columns={"observation_date": "date", "QUSR628BIS": "real_property_price"})
prop["date"] = to_quarter_end(prop["date"])
prop["date"] = prop["date"] + pd.offsets.QuarterEnd(1)
prop = prop.groupby("date").agg(real_property_price=("real_property_price", "last")).reset_index()
merged = pd.merge(merged, prop, on="date", how="left")

# --- UNRATE (monthly → quarterly average) ---
unrate = pd.read_csv("data/macro/UNRATE.csv", parse_dates=["observation_date"])
unrate = unrate.rename(columns={"observation_date": "date", "UNRATE": "unrate"})
unrate["date"] = to_quarter_end(unrate["date"])
unrate = unrate.groupby("date").agg(unrate=("unrate", "mean")).reset_index()
merged = pd.merge(merged, unrate, on="date", how="left")

merged = merged.sort_values("date").reset_index(drop=True)

In [98]:
print(merged.shape)
print(merged.isnull().sum())

(44, 39)
date                          0
YLDMAT                        0
DAYMAT                        0
b30ret                        0
b30ind                        0
b20ret                        0
b20ind                        0
b10ret                        0
b10ind                        0
b7ret                         0
b7ind                         0
b5ret                         0
b5ind                         0
b2ret                         0
b2ind                         0
b1ret                         0
b1ind                         0
t90ret                        0
t90ind                        0
t30ret                        0
t30ind                        0
cpiret                        0
cpiind                        0
wti_price                     0
trade_balance                 0
eci                           0
gdp                           0
gscpi                         0
long_term_rate                0
median_cpi                    0
nfci                          0

In [ ]:
merged.to_csv("data/macro/combined_macro_data.csv", index=False)
print(f"Saved: data/macro/combined_macro_data.csv")
print(f"Shape: {merged.shape[0]} rows × {merged.shape[1]} columns")
print(f"Date range: {merged['date'].min().date()} → {merged['date'].max().date()}")
print(f"\nColumns:\n{list(merged.columns)}")